In [1]:
import os
import os.path as op
import argparse
from stress_risk.utils.data import Subject
import numpy as np
from braincoder.utils import get_rsq
import pandas as pd
from nilearn.maskers import NiftiMasker
from nilearn import image

/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
subject = 58 # copied all necessary files to local 
bids_folder='/Users/mrenke/data/ds-stressrisk'
sub = Subject(subject,bids_folder=bids_folder)

model_label = 6
regressor_name = 'session'
if model_label == 6: ## this! -- we dont have a prediction on how exaclty the mu changes! 
    regressors = {'mu':f'0 + C({regressor_name})'}
    
gaussian = True
retroicor = True
roi='NPC_R'

source_key_glm = 'glm_stim1.denoise'
source_key_vselect = 'encoding_model.cv.denoise'



In [31]:
target_key = 'encoding_model'
target_key += f'.model{model_label}'

if gaussian:
    target_key += '.gaussian'
else:
    target_key += '.logspace'

if retroicor:
    target_key += '.retroicor'
    source_key_glm += '.retroicor'
    source_key_vselect += '.retroicor'

target_dir = op.join(bids_folder, 'derivatives', target_key, f'sub-{subject}', 'func')
if not op.exists(target_dir):
    os.makedirs(target_dir)

print(target_dir)

/Users/mrenke/data/ds-stressrisk/derivatives/encoding_model.model6.gaussian.retroicor/sub-58/func


In [4]:
from stress_risk.utils.data import Subject

sub = Subject(subject, bids_folder=bids_folder)
behavior = sub.get_behavior(sessions=None).reset_index('session') # session will be range
#behavior.head()
paradigm = behavior[['n1', 'session']].rename(columns={'n1':'x' }) #,'session':'range'
if not gaussian:
    paradigm['x'] = np.log(paradigm['x'])
#paradigm['session'] = (paradigm['session'] == 2)
paradigm['x'] = paradigm['x'].astype(np.float32)

paradigm



x  session
subject run trial_nr                
58      1   1           7.0        1
            2           7.0        1
            3          28.0        1
            4          20.0        1
            5          10.0        1
...                     ...      ...
        6   116        12.0        2
            117       105.0        2
            118        45.0        2
            119        14.0        2
            120        35.0        2

[240 rows x 2 columns]

In [5]:
# mask: only take voxels which were also used for neural-code-shift (=encoding_model.cv.denoise/../ses-1; vor CV voxel selection) | just like in decode_svoxels.py (across session decoding!)
#masker = sub.get_brain_mask(session=None, epi_space=True, return_masker=True, debug_mask=debug)
# get average cv-r2 map from seesion 1 for voxel selection
session1 = 1
ips_mask = sub.get_volume_mask(roi=roi, session=1, epi_space=True) # anat from session1
ips_masker = NiftiMasker(mask_img=ips_mask)

im_cvr2_fn = op.join(bids_folder, 'derivatives', source_key_vselect, f'sub-{subject}', f'ses-{session1}','func', f'sub-{subject}_ses-{session1}_desc-cvr2.optim_space-T1w_pars.nii.gz')
im_cvr2 = image.load_img(im_cvr2_fn)
cv_r2 = pd.DataFrame(ips_masker.fit_transform(im_cvr2))
r2_mask = cv_r2 > 0.0
r2_mask = r2_mask.to_numpy().T

masker = NiftiMasker(mask_img=r2_mask)
n_voxels = r2_mask.sum()

/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/image/image.py:756: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  return klass(data, affine, header=header)


In [6]:
# single trial functional brain data
data_s1 = op.join(bids_folder, 'derivatives', source_key_glm,
                f'sub-{subject}', f'ses-1', 'func', f'sub-{subject}_ses-1_task-risk_space-T1w_desc-stims1_pe.nii.gz')
data_s2 = op.join(bids_folder, 'derivatives', source_key_glm,
                f'sub-{subject}', f'ses-2', 'func', f'sub-{subject}_ses-2_task-risk_space-T1w_desc-stims1_pe.nii.gz')

data = np.concatenate([ips_masker.fit_transform(data_s1), ips_masker.fit_transform(data_s2)], axis=0)
data = pd.DataFrame(data, index=paradigm.index)
data

0         1         2         3         4    \
subject run trial_nr                                                     
58      1   1         1.360907  0.984276  0.469469  1.296231 -1.777357   
            2         0.038862 -0.032263 -0.511647  0.696825 -0.671320   
            3        -1.562327  0.022479  1.211599 -0.486628  1.791842   
            4        -0.285637 -0.469450 -0.937121  0.370980 -0.433528   
            5         1.658858 -0.736334  0.097248  0.330500 -1.636484   
...                        ...       ...       ...       ...       ...   
        6   116       0.267398  0.037879  0.373627  1.834324  0.849786   
            117      -0.497814 -0.193648  0.563297 -2.997992 -1.719001   
            118      -0.792068  0.191731  0.402316  0.657624 -1.419436   
            119       2.506625 -0.244071  0.009343  0.457890 -0.251963   
            120      -0.352203  0.892210  0.185911 -0.150341 -0.448944   

                           5         6         7         8         9    ...  \
subject run trial_nr                                                    ...   
58      1   1         0.022169 -1.943530  2.437336 -0.677708 -0.617421  ...   
            2         0.933234 -0.843968 -0.121207 -1.179444 -1.278525  ...   
            3         0.200672 -0.303261 -3.081260  0.643819 -0.824857  ...   
            4        -0.931633  0.466594  0.076802 -0.955287 -0.686287  ...   
            5        -1.270384 -0.002720  2.163406 -1.017937  0.529069  ...   
...                        ...       ...       ...       ...       ...  ...   
        6   116      -0.268588  0.309693 -0.452175 -0.677455 -2.209673  ...   
            117      -0.453025 -0.027860 -1.323911 -0.218892 -0.901782  ...   
            118      -1.005738 -1.437503  0.395998 -0.189040 -0.094481  ...   
            119       0.189522  1.126184 -0.741739  0.717575 -0.741771  ...   
            120      -1.078865  0.125243  3.172607 -1.206912 -0.857233  ...   

                           882       883       884       885       886  \
subject run trial_nr                                                     
58      1   1         0.115563  0.205481 -1.597418 -0.461507 -0.745871   
            2         0.025700 -0.270255  0.710059  0.753099  0.759857   
            3         0.697430 -0.586619 -0.428200 -0.043618  0.359179   
            4        -0.976089  0.300377  0.884656  0.456068  0.231440   
            5        -0.255290 -0.700215  0.128976  0.237128 -0.598346   
...                        ...       ...       ...       ...       ...   
        6   116      -0.285128  0.272037 -0.988437  1.369822 -0.148745   
            117       0.683748 -0.881824  0.465243  0.957454  0.462458   
            118      -0.076281 -2.525728  0.413944  0.028813  0.432050   
            119       0.083669 -0.605084 -0.124368  2.037228  1.187818   
            120       0.091553  0.988466  1.290519  0.528098  0.449285   

                           887       888       889       890       891  
subject run trial_nr                                                    
58      1   1        -0.529340  0.020227  0.001107 -0.840217 -0.262518  
            2         0.936433  0.654427 -0.173055  0.429614 -0.847238  
            3         0.688186 -0.190014 -0.570956  0.170432 -0.494595  
            4         0.133600  1.019008 -0.240482  0.058865 -0.875100  
            5        -0.227395 -0.734497 -0.748980  0.435837  0.043196  
...                        ...       ...       ...       ...       ...  
        6   116       0.142943  1.380131  1.880349  0.658388  0.941032  
            117      -0.142516  1.023011  2.302685 -0.668543 -0.347593  
            118       0.147773  0.545773 -0.137250 -0.928068 -0.340507  
            119      -0.627450 -0.529821 -1.919407 -0.364926 -0.072064  
            120       0.871795 -0.189508 -0.204360  0.753086 -0.310534  

[240 rows x 892 columns]

In [ ]:
# Get model
from braincoder.models import RegressionGaussianPRF

#model = get_model(paradigm, model_label, gaussian=gaussian)
model = RegressionGaussianPRF(paradigm=paradigm, regressors=regressors) # should match with sesssion as regressor!

# Fit model
from models_sessionRegressor import fit_model, get_conditionspecific_parameters
debug = True
max_n_iterations = 100 if debug else 1000
pars = fit_model(model, paradigm, data, model_label, max_n_iterations=max_n_iterations, gaussian=gaussian)

pred = model.predict(parameters=pars, paradigm=paradigm)
r2 = get_rsq(data, pred)

# takes ~ 65 min

2025-03-05 08:08:08.420364: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-03-05 08:08:08.421035: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


                          x  session
subject run trial_nr                
58      1   1           7.0        1
            2           7.0        1
            3          28.0        1
            4          20.0        1
            5          10.0        1
...                     ...      ...
        6   116        12.0        2
            117       105.0        2
            118        45.0        2
            119        14.0        2
            120        35.0        2

[240 rows x 2 columns]
FITTING GRID
Working with chunk size of 3114
Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB



  0%|          | 0/2 [00:00<?, ?it/s]2025-03-05 08:08:08.858480: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2025-03-05 08:08:08.863209: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
100%|██████████| 2/2 [00:05<00:00,  2.66s/it]


parameter  mu_unbounded               sd_unbounded amplitude_unbounded  \
regressor C(session)[1] C(session)[2]    Intercept           Intercept   
count        892.000000    892.000000   892.000000               892.0   
mean          34.443981     32.970116     4.281871                 1.0   
std           10.489923     14.580482     3.292519                 0.0   
min            5.000000      5.000000     3.000000                 1.0   
25%           31.666666     18.333334     3.000000                 1.0   
50%           37.000000     42.333332     3.000000                 1.0   
75%           42.333332     45.000000     3.000000                 1.0   
max           45.000000     45.000000    15.000000                 1.0   

parameter baseline_unbounded  
regressor          Intercept  
count                  892.0  
mean                     0.0  
std                      0.0  
min                      0.0  
25%                      0.0  
50%                      0.0  
75%        

  0%|          | 0/100 [00:00<?, ?it/s]2025-03-05 08:08:14.325935: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2025-03-05 08:08:16.543549: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
Current R2: 0.02263/Best R2: 0.02263: 100%|██████████| 100/100 [31:12<00:00, 18.72s/it] 
2025-03-05 08:39:26.014655: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


Number of problematic voxels (mask): 0
Number of voxels remaining (mask): 892


  0%|          | 0/100 [00:00<?, ?it/s]2025-03-05 08:39:28.080155: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2025-03-05 08:39:30.103693: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
Current R2: 0.02788/Best R2: 0.02788: 100%|██████████| 100/100 [30:56<00:00, 18.57s/it]


parameter  mu_unbounded               sd_unbounded amplitude_unbounded  \
regressor C(session)[1] C(session)[2]    Intercept           Intercept   
count        892.000000    892.000000   892.000000          892.000000   
mean          34.385113     33.032009     3.837150           -0.143351   
std           10.715002     14.817999     3.375936            0.861866   
min            3.902640      3.768816     1.700121           -2.494700   
25%           30.800799     18.006043     2.044495           -0.523357   
50%           36.243553     41.417089     2.323103           -0.099202   
75%           42.859864     45.453088     3.780438            0.345033   
max           46.202839     46.295193    15.998238            3.948309   

parameter baseline_unbounded  
regressor          Intercept  
count             892.000000  
mean                0.018327  
std                 0.265182  
min                -1.155515  
25%                -0.132333  
50%                -0.003647  
75%        

In [ ]:
target_fn = op.join(target_dir, f'sub-{subject}_desc-r2.optim_space-T1w_pars.npy')
np.save(target_fn, np.array(r2))
#ips_masker.inverse_transform(r2).to_filename(target_fn)


problem: did not pull latest braincode, hence model does not have `get_conditionspecific_parameters`

try workaround:


In [23]:
regressor_name = 'session'
parameters = pars.copy()

conditions = pd.DataFrame({'x':[0,0], regressor_name:[0,1]}, index=pd.Index(['1', '2'], name=regressor_name))

design_matrices = model.build_design_matrices(conditions,regressors=regressors ) #cond = paradigm ?

if hasattr(parameters, 'values'):
    parameters_ = parameters.values
else:
    parameters_ = np.array(parameters)

parameters_ = parameters_[np.newaxis, ...]

transformed_parameters = model._get_base_parameters(design_matrices, parameters_).numpy()

transformed_parameters = np.reshape(transformed_parameters, (-1, transformed_parameters.shape[-1]))

transformed_parameters = pd.DataFrame(transformed_parameters,
                            index=pd.MultiIndex.from_product([conditions.index, parameters.index]),
                            columns=model.base_parameter_labels)

2025-03-05 10:10:43.422646: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


In [24]:
transformed_parameters

mu        sd  amplitude  baseline
session source                                          
1       0       42.054165  3.562764   0.683829 -0.238515
        1       45.612434  3.079819   0.470613 -0.163835
        2       42.702728  2.225185   0.981917  0.019922
        3       45.936527  3.014135   0.966498 -0.108692
        4       36.319279  3.916447   0.472458 -0.532330
...                   ...       ...        ...       ...
2       887     33.237793  2.107573   0.794459 -0.120282
        888     42.080803  2.207717   0.378755  0.201911
        889     38.889774  2.175251   0.279232 -0.170330
        890     41.653259  3.929009   0.243035 -0.139884
        891     34.714848  2.509721   0.678708 -0.008026

[1784 rows x 4 columns]

In [ ]:
for regres_vars, values in transformed_parameters.groupby(regressor_name):
    for par, value in values.T.iterrows():
        target_fn = op.join(target_dir, f'sub-{subject}_desc-{par}.{regres_vars}.optim_space-T1w_pars.npy')
        #masker.inverse_transform(value).to_filename(target_fn)
        np.save(target_fn, np.array(value))



In [27]:
target_fn

'/Users/mrenke/data/ds-stressrisk/derivatives/encoding_model.model6.gaussian.retroicor/sub-58/func/sub-58_desc-baseline.2.optim_space-T1w_pars.npy'

In [19]:
os.makedirs('/Users/mrenke/data/ds-stressrisk/derivatives/fmriprep/sub-58/ses-1/func/')